# How to do Backpropagation with numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time

## Layer class

Like they provide a *forward* function, all our different layer types will get an *backward* function as well. Here we pass the upstream gradient as well as the original inputs to the function. Based on this the Layer will calculate the downstream gradient. 

In [ ]:
class Layer:
    """
    An identity Layer to be the base of inheritance
    """
    def __init__(self):
        pass

    # input.shape = (samples, features)
    # returnvalue.shape = (samples, features)
    def forward(self, input):
        # identity, same output as input
        return input

    # upstream_gradient: we expect the gradient of the loss with respect to the output of the layer
    # upstream_gradient.shape = (samples, #nodes in current layer)
    def backward(self, input, upstream_gradient, verbose=False):
        # The gradient of a dummy layer is precisely grad_output, but we'll write it more explicitly
        return upstream_gradient * 1

## Updating the Sigmoid Function 

The derivative of the function

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\sigma(x) = \frac{1}{1+e^{-x}}$ 

is 

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\sigma'(x) = \sigma(x) \cdot (1-\sigma(x))$

The chain rule says:
$\frac{\partial L}{\partial input} = \frac{\partial L}{\partial output} \cdot \frac{\partial output}{\partial input} = upstream gradient * local gradient$ 

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def derivative_sigmoid(x):
    return sigmoid(x)*(1- sigmoid(x))

class Sigmoid(Layer):
    def __init__(self):
        pass
    
    def forward(self, input):
        return sigmoid(input)
    
    def backward(self, input, upstream_gradient, verbose=False):
        # apply the cain rule
        local_gradient = derivative_sigmoid(input)
        downstream_gradient = upstream_gradient * local_gradient
        return downstream_gradient
    
    def __str__(self):
        return "Sigmoid()"

Out input can be a vector or a matrix. Usually it will be a matrix of $batchsize \times inputnodes$. Thus the gradients will be of size $batchsize \times outputnodes$.

# Implmenting a RELU Layer

ReLU is similar to $max$ regarding backpropagation. We had the max function on the lecture slides.

In [ ]:
class ReLU(Layer):
    def __init__(self):
        # ReLU layer simply applies elementwise rectified linear unit to all inputs
        pass
    
    def forward(self, input):
        # Apply elementwise ReLU to [batch, input_units] matrix
        return np.maximum(0,input)
    
    def backward(self, input, grad_output):
        # Compute local gradient of loss w.r.t. ReLU input
        local_gradient = input > 0
        upstream_gradient = grad_output
        return upstream_gradient*local_gradient

# Backward function for the Dense Layer

In the activation functions we just backpropagated gradients, we had no learnable parameters (weights or biases) to update. This time we do more: 
* first we calculate gradients with respect to inputs (weights fixed)
* then we calculate the gradient with respect to weights (inputs fixed)
* we calculate the gradient with respect to the bias values (inputs fixed)
* Then weights and biases are updated considering gradients and learning rate
* Finally we return the gradients with respect to the insputs as the new downstream gradients

In [ ]:
class Dense(Layer):
    def __init__(self, input_units, output_units, learning_rate):
        # initialize weights with small random numbers. We use a normal distribution 
        self.weights = np.random.randn(input_units, output_units)*0.01
        self.biases = np.zeros(output_units)
        self.learning_rate=learning_rate

    #   (samples , inputunits) \dot (inputsunits, outputunits)
    #   dimension of inputs = (samples , inputunits)
    def forward(self,input):
        return input.dot(self.weights)+self.biases;

    # dimensions of grad_input = (samples ,outputs) * (outputs, inputs)
    # dimensions of grad_weights = (inputs, samples) * (samples ,outputs) = (inputs, outputs)
    # dimensions of grad_biases = (outputs, )
    def backward(self,input,grad_output,verbose=False):
        #compute downstream gradient = gradient with respect to inputs
        grad_input = np.dot(grad_output, self.weights.T) 
        
        #compute gradient with respect to weights and biases
        grad_weights = np.dot(input.T,grad_output)/input.shape[0] 
        grad_biases = grad_output.mean(axis=0) 
        
        assert grad_weights.shape == self.weights.shape and grad_biases.shape == self.biases.shape
        
        #now we can sounchronuously update the weights = one stochastic gradient descent step. 
        self.weights = self.weights - self.learning_rate*grad_weights
        self.biases = self.biases - self.learning_rate*grad_biases
        
        return grad_input
    
    def __str__(self):
        text = f"Dense Layer {self.weights.shape}"
        return text        

## Integrating a gradient descent step to the network

Our first network shall just be able to learn the "and"-Function for binary values (0,1). So it can be rather small with just one layer so its easy to debug. Later we will increase the number of layers.

* During forward we will store all the inputs/outputs between the layers
* A dedicated predict function returns the outputs of the last layer
* We have to implement the derivative of the loss function
* We implement "trainBatch" which does one gradient descent step 

In [ ]:
class Network:
    def __init__(self, learning_rate, iterations, logcount=1): 
        self.layers = []
        self.layers.append(Dense(2,1, learning_rate))
        self.layers.append(Sigmoid())
        print("Number of layers", len(self.layers))
        self.iterations = iterations
        self.logcount = logcount
        
    def __str__(self):
        text = "Network:\n"
        for lay in self.layers:
            text = text + "  " +lay.__str__() + "\n"
        return text

    # X is a matrix of trainingdata (samples, features)
    # In contrast to the old implementation we return the result of all layers 
    # and not just the last
    def forward(self, X):
        activations = [X]
        for layer in self.layers:
            X = layer.forward(X);
            activations.append(X)
        assert len(activations) == len(self.layers)+1
        #print("activations.shape", activations.shape)
        return activations

    # the activations of the last (index==-1) layer are returned
    def predict(self, input, verbose=False):
        return self.forward(input)[-1]

    def loss(self, y, pred):
        assert y.shape == pred.shape
        vec = -np.log(pred)*y-(np.log(1-pred)*(1-y))
        return np.mean(vec); # Aggregartion over all samples
    
    # https://stats.stackexchange.com/questions/428228/how-does-cross-entropy-log-loss-work-with-backpropagation
    def log_loss_prime(self, y_true, y_pred):
        return ((1 - y_true) / (1 - y_pred) - y_true / y_pred) / np.size(y_true)

    # Forward pass -> inputs of all layers are stored
    # We compute the prediction and the loss (which will be returned)
    # Based on the loss we compute the upstream gradient for the last layer
    # Layer by layer we call the backward function
    #  - the input is passed in to compute the local gradient
    #  - the upstream gradient is passed in
    #  - inside the backward function the weights are updated
    #  - the return value is the downstream gradient for the preceding layer
    #
    # In summary:
    #    for each (X, y) call, the weights are nudged slightly so that the loss decreases
    def trainBatch(self, X, y, verbose = False):
        layer_inputs = self.forward(X)
        pred = layer_inputs[-1];
        if verbose: 
            print("layer_inputs")
            for lay in layer_inputs:
                print(lay)
            print("predictions", pred)

        # Compute the loss and the initial gradient
        loss_grad=self.log_loss_prime(y,pred)
        if verbose: print("Shape ", loss_grad.shape, " Value:", loss_grad)
    
        index = -2;
        for layer in reversed(self.layers):
            loss_grad = layer.backward(layer_inputs[index],loss_grad) #grad w.r.t. input, also weight updates
            if (verbose):
                print("Layer:", index, layer);
                print("Loss_grad:", loss_grad);

            index = index -1
        
        return self.loss(y, pred);
    
    def fit(self, X,y, verbose = False):
        if (len(y.shape)==1):
            y.reshape(-1,1)
            print("Reshaped groud truth to ", y.shape)
        
        for i in range(self.iterations):
            if i%self.logcount==0:
                print(self.trainBatch(X,y, verbose))
            else:
                self.trainBatch(X,y, verbose)

## Make a prediction with an untrained model 

In [ ]:
net = Network(0.1, 100000,10000)
#Trainingsamples
input = np.array([
    [1.0,1.0],
    [1.0,0.0],
    [0.0,1.0],
    [0.0,0.0],
]) 
y = np.array([1.0,0.0,0.0,0.0]).reshape(4,1)
pred = net.predict(input)
print("prediction",pred)
assert y.shape == pred.shape
print("loss",net.loss(y,pred))


## Train model

In [ ]:
net.fit(input,y)

## Predict again with trained weights

In [ ]:
pred = net.predict(input)
print("prediction",np.round(pred,2)) 
assert y.shape == pred.shape
print("loss",net.loss(y,pred))

In [ ]:
X = np.random.rand(800,2)
y = net.predict(X)>0.5
import matplotlib.pyplot as plt
plt.scatter(X[:,0],X[:,1],s=2,c=y)

# Train with continuous values and create nonlinear decision border

In [ ]:
class DeepNetwork(Network):
    def __init__(self, learning_rate, iterations, logcount=1):
        super().__init__(learning_rate, iterations, logcount)
        # Override the layers with the deeper architecture
        self.layers = []
        self.layers.append(Dense(2,2, learning_rate))
        self.layers.append(Sigmoid())
        self.layers.append(Dense(2,1, learning_rate))
        self.layers.append(Sigmoid())
        print("Number of layers", len(self.layers))

In [ ]:
X = np.random.rand(800,2)
y = ((X[:,0]>0.5)*1.0+(X[:,1]>0.5)*1.0)==2  # determines ground truth value
y=y.reshape(800,1)
plt.scatter(X[:,0],X[:,1],s=2,c=y)
plt.title("Training Data")

In [ ]:
net = DeepNetwork(100,100000,10000)
net.fit(X,y)

In [ ]:
# what has been learned
p = net.predict(X)>0.5
plt.scatter(X[:,0],X[:,1],s=2,c=p)

# Exercises

* What happens if we remove the first sigmoid layer ? Guess, try and explain!
* Try to plot a curve and vizualize how the loss drops! (This is called training curve)
* Experiment with the learning-rate and iterations, how sensible is the algorithm to them
* Create XOR data (y = ((X[:,0]>0.5)*1.0+(X[:,1]>0.5)*1.0)==1  # XOR: exactly one feature > 0.5)
  * Will the current network be able to learn it ?
  * Adapt the network and train it ?
* After Training vizualize results of activations in the intermediate layers